# Soft Actor-Critic (SAC) for Connect-4 — PG + Q-learning fused

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Stiles-Clements1/connect4-rl-arena/blob/re_org/notebooks/sac_training.ipynb)

Three networks, one agent:

- **Policy π(a|s)** — 7-column softmax (the *policy gradient* side)
- **Q(s, a)** — 7-column linear, trained via DQN-style Bellman targets (the *Q-learning* side)
- **Target Q network** — polyak-updated snapshot for stable Bellman bootstrapping

This is the real fusion of the two methods the assignment covers.

Losses:

    Q-loss     = MSE( Q(s, a), r + γⁿ(1-done) Σ_a' π(a'|s') [Q_target(s', a') - α log π(a'|s')] )
    π-loss     = E_s [ Σ_a π(a|s) (α log π(a|s) - Q(s, a)) ]
    α (entropy temperature) is fixed.

**Runs on both Colab and local.** Cell 1 below detects the environment
automatically (clones the repo on Colab, locates the repo root locally)
— no code edits needed. The checkpoint committed at `checkpoints/sac_run/`
means re-running Cell 7 on a fresh Colab picks up training from our
latest saved model.

Full training notes, methodology and results are in
[`logs/sac_training_notes.md`](../logs/sac_training_notes.md).

In [ ]:
# Cell 1 — Environment setup (Colab OR local, no edits needed).
#
# Detects whether we're running in Google Colab. On Colab, clones the re_org branch
# of the team repo (or fetches and resets to it if the clone already exists).
# On local runs, walks up from the notebook's CWD to find the repo root.
# Either way, REPO_ROOT points at the repo, src/ is importable, and the
# working directory is set to the repo root so relative paths like 'logs/'
# and 'checkpoints/' resolve correctly.
#
# After `re_org` merges into `main`, flip GITHUB_BRANCH back to 'main'.

import os
import sys
import subprocess
import warnings
from pathlib import Path

warnings.filterwarnings('ignore', message=r'.*structure of `inputs`.*')

IS_COLAB       = 'google.colab' in sys.modules
GITHUB_URL     = 'https://github.com/Stiles-Clements1/connect4-rl-arena.git'
GITHUB_BRANCH  = 'main'

if IS_COLAB:
    REPO_ROOT = Path('/content/connect4-rl-arena')
    if not REPO_ROOT.exists():
        print(f'Cloning {GITHUB_URL} (branch {GITHUB_BRANCH}) → {REPO_ROOT}')
        subprocess.run(
            ['git', 'clone', '--branch', GITHUB_BRANCH, GITHUB_URL, str(REPO_ROOT)],
            check=True,
        )
    else:
        print(f'Repo already at {REPO_ROOT}; fetching + checking out {GITHUB_BRANCH}.')
        subprocess.run(['git', '-C', str(REPO_ROOT), 'fetch', '--quiet', 'origin'], check=False)
        subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout', '--quiet', GITHUB_BRANCH], check=False)
        subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--quiet', 'origin', GITHUB_BRANCH], check=False)
    subprocess.run(['pip', 'install', '-q', 'tqdm', 'ipywidgets'], check=False)
else:
    # Local: walk up from the notebook's CWD looking for the repo root.
    # If we don't find it (e.g. the notebook was downloaded standalone for
    # grading), clone the repo into ./connect4-rl-arena so this notebook
    # is fully self-bootstrapping with just internet + git + python.
    here = Path.cwd().resolve()
    REPO_ROOT = None
    for p in [here, *here.parents]:
        if (p / 'src' / 'game_engine.py').exists():
            REPO_ROOT = p
            break
    if REPO_ROOT is None:
        REPO_ROOT = here / 'connect4-rl-arena'
        if not REPO_ROOT.exists():
            print(f'No repo found in directory tree from {here}.')
            print(f'Cloning {GITHUB_URL} (branch {GITHUB_BRANCH}) → {REPO_ROOT}...')
            subprocess.run(
                ['git', 'clone', '--branch', GITHUB_BRANCH, GITHUB_URL, str(REPO_ROOT)],
                check=True,
            )
        else:
            print(f'Repo already at {REPO_ROOT}; fetching + checking out {GITHUB_BRANCH}.')
            subprocess.run(['git', '-C', str(REPO_ROOT), 'fetch',  '--quiet', 'origin'], check=False)
            subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout','--quiet', GITHUB_BRANCH], check=False)
            subprocess.run(['git', '-C', str(REPO_ROOT), 'pull',   '--quiet', 'origin', GITHUB_BRANCH], check=False)
    else:
        print(f'Local repo root: {REPO_ROOT}')

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

# GPU init BEFORE importing model_loader (so TF places models on GPU).
import tensorflow as tf
_gpus = tf.config.list_physical_devices('GPU')
for _g in _gpus:
    try:
        tf.config.experimental.set_memory_growth(_g, True)
    except Exception:
        pass
HARDWARE = f'GPU ({_gpus[0].name.split(":")[-1]})' if _gpus else 'CPU only'
print(f'Hardware = {HARDWARE}   |   TF = {tf.__version__}')
print(f'IS_COLAB = {IS_COLAB}   |   branch = {GITHUB_BRANCH}   |   cwd = {REPO_ROOT}')

# Purge cached src.* imports so pulls mid-session pick up fresh code.
for _m in [k for k in sys.modules if k == 'src' or k.startswith('src.')]:
    del sys.modules[_m]

from src import model_loader
from src.sac_trainer import (
    SACConfig, build_sac_model, build_sac_from_pretrained,
    train, SACAgent, MinimaxOpponentWrapper,
)
from src.eval import ModelAgent, RandomAgent, MinimaxAgent

models = model_loader.load_all_models()
print(f'\nLoaded {len(models)} pretrained models.')


In [ ]:
# Cell 2 — Initial opponent pool.
#
# The SAC agent learns against diverse opponents: all pretrained Project-1
# networks PLUS calibrated minimax players at moderate depth.
#
# `luke_transformer` is excluded because its round-robin performance (~20%)
# suggests either a loading bug or a genuinely broken model; training
# against it teaches exploits that don't generalise.
#
# minimax_d5 is INTENTIONALLY omitted from training — it wins ~98% against
# a warm-started SAC, so the all-loss signal collapses the critic. Kept
# as a BENCHMARK below so we can still measure progress vs it.

EXCLUDE = {'luke_transformer'}

initial_pool = [w for k, w in models.items()
                if k != 'm1' and k not in EXCLUDE]
initial_pool.insert(0, models['m1'])

initial_pool.append(MinimaxOpponentWrapper(depth=1))
initial_pool.append(MinimaxOpponentWrapper(depth=3))

print(f'Opponent pool has {len(initial_pool)} opponents:')
for w in initial_pool:
    print(f'  - {w.name}  (encoding {w.encoding})')

In [ ]:
# Cell 3 — Benchmarks for periodic in-training evaluation.
#
# These are run every `cfg.eval_interval` groups during training, so we
# can see progress without waiting for the full run to finish.

benchmarks = {
    'stiles_transformer_orig': ModelAgent(models['stiles_transformer_orig'],
                                          greedy=True, use_tactics=True),
    'stiles_cnn':               ModelAgent(models['stiles_cnn'],
                                          greedy=True, use_tactics=True),
    'minimax_d3':               MinimaxAgent(depth=3),
    'random':                   RandomAgent(),
}
print(f'Benchmarks: {list(benchmarks.keys())}')

In [ ]:
# Cell 4 — Warm-start SAC from a pretrained policy network.
#
# The SAC policy head is the pretrained network's 7-column softmax output
# (weights preserved). The Q head is BRAND NEW (random init) and hooks
# off the pretrained penultimate layer so actor and critic share a trunk.
# This cuts 'time to first win' dramatically because the policy starts at
# a strong supervised baseline instead of at random.
#
# Set WARM_START_FROM to None for a tiny ~3 MB model trained from scratch
# (slower to converge, but minimal disk footprint).

WARM_START_FROM = 'zan_cnn'    # options: 'zan_cnn', 'stiles_cnn',
                               # 'stiles_transformer_orig',
                               # 'zan_transformer', 'luke_cnn', or None

if WARM_START_FROM is None:
    initial_sac_model = None
    print('No warm start — SAC will build a small fresh network from scratch.')
elif WARM_START_FROM in models:
    initial_sac_model = build_sac_from_pretrained(
        models[WARM_START_FROM].model, q_hidden=256,
    )
    print(f'\nWarm-started from {WARM_START_FROM}. '
          f'Trainable params: {initial_sac_model.count_params():,}')
else:
    # Cascading fallback if the requested warm-start model is unavailable
    for alt in ('stiles_cnn', 'stiles_transformer_orig', 'luke_cnn', 'zan_cnn'):
        if alt in models:
            initial_sac_model = build_sac_from_pretrained(
                models[alt].model, q_hidden=256,
            )
            print(f'\n(requested {WARM_START_FROM!r} unavailable — using {alt} instead)')
            break
    else:
        initial_sac_model = None
        print('\nNo pretrained network available — SAC will build from scratch.')

In [ ]:
# Cell 5 — Hyperparameters.
#
# CHECKPOINT_DIR is where training auto-saves every `checkpoint_interval`
# groups. Re-running Cell 7 with the same path resumes cleanly from the
# committed checkpoint. All values below reflect the final submission run
# (see logs/sac_training_notes.md for the tuning story).

CHECKPOINT_DIR = str(REPO_ROOT / 'checkpoints' / 'sac_run')

cfg = SACConfig(
    num_groups         = 1000,
    games_per_group    = 128,
    updates_per_group  = 16,
    batch_size         = 256,
    min_buffer_size    = 2000,

    learning_rate      = 1e-4,   # lowered from 3e-4 for conservative updates
    gamma              = 0.98,

    # Entropy temperature. α=0.1 pulled the warm-started policy toward
    # uniform and destroyed skill before Q was accurate enough to guide
    # the policy back. α=0.03 keeps some exploration without un-sharpening
    # the warm start. Standard SAC-discrete range is 0.01-0.05.
    alpha              = 0.03,
    tau                = 0.005,  # soft target update rate (Polyak)

    buffer_capacity    = 50000,

    shaping_coef       = 0.05,
    tactics_prob       = 0.25,   # fewer training games vs tactically-enhanced M2

    pool_cap              = 12,
    pool_add_interval     = 30,
    pool_originals_weight = 3.0,
    pool_snapshot_weight  = 1.0,

    q_hidden           = 256,
    conv_filters       = (64, 128, 128),
    dense_units        = 256,
    dropout_rate       = 0.1,

    checkpoint_dir       = CHECKPOINT_DIR,
    checkpoint_interval  = 20,
    resume_if_exists     = True,

    eval_interval        = 25,
    eval_n_games         = 40,
)
for k, v in cfg.__dict__.items():
    print(f'  {k:<24s} {v}')

In [ ]:
# Cell 6 — Sanity probe: is the warm-started policy competitive?
#
# Runs a quick 50-game head-to-head between the freshly-built (un-trained)
# SAC model and stiles_transformer_orig. Expected: 40-55% win rate from
# the warm-start inheritance. If this prints ~0%, something is broken
# and more training will not help.
#
# Skipped when we're about to resume from a checkpoint — the probe is
# only meaningful for a fresh start.

_state_path = Path(CHECKPOINT_DIR) / 'state.json'
_will_resume = _state_path.exists()

if initial_sac_model is not None and not _will_resume:
    from src.eval import play_match_parallel

    _probe_agent  = SACAgent(initial_sac_model, name='sac_pretrained_probe',
                             greedy=True, use_tactics=True)
    _stiles_probe = ModelAgent(models['stiles_transformer_orig'],
                               greedy=True, use_tactics=True)
    _probe = play_match_parallel(_probe_agent, _stiles_probe,
                                 n_games=50, random_init_moves=4, progress=False)
    print(f'Pre-training probe (warm-started SAC vs stiles_transformer_orig):')
    print(f'  Win rate: {_probe.a_win_rate:.0%}  Draw: {_probe.draw_rate:.0%}  Loss: {_probe.b_win_rate:.0%}')
    if _probe.a_win_rate < 0.15:
        print('  ⚠ WARNING: warm-start looks broken.')
    else:
        print('  ✓ Warm-start healthy.')
elif _will_resume:
    print(f'Found existing checkpoint at {CHECKPOINT_DIR} — skipping warm-start probe.')
    print('Cell 7 will resume from the checkpoint and continue training.')
else:
    print('No initial_sac_model — training from scratch will be slow to bootstrap.')

In [ ]:
# Cell 7 — TRAIN (idempotent — resumes from checkpoint if present).
#
# Each invocation of this cell adds ADDITIONAL_GROUPS more training groups
# on top of whatever checkpoint already exists at CHECKPOINT_DIR. Safe to
# re-run as many times as you want — each run banks more training.
#
#   First run (no checkpoint)  : warm-start from initial_sac_model,
#                                 train ADDITIONAL_GROUPS groups.
#   Re-runs (checkpoint exists): RESUME from it, train another
#                                 ADDITIONAL_GROUPS groups on top.
#
# To start completely fresh: delete the CHECKPOINT_DIR folder first.

ADDITIONAL_GROUPS = 500   # each run of this cell trains this many more groups

_state_path = Path(CHECKPOINT_DIR) / 'state.json'

if _state_path.exists():
    import json as _json
    with open(_state_path) as _f:
        _group_done = _json.load(_f)['group_done']
    cfg.num_groups = _group_done + ADDITIONAL_GROUPS
    _init_for_train = None                    # resume — don't overwrite
    print(f'Resuming SAC at group {_group_done}; training to {cfg.num_groups} '
          f'(+{ADDITIONAL_GROUPS} groups).')
else:
    cfg.num_groups = ADDITIONAL_GROUPS
    _init_for_train = initial_sac_model       # first run — warm start
    print(f'Fresh start — training {ADDITIONAL_GROUPS} groups from the warm start. '
          f'Future runs of this cell will resume and add more.')

result = train(
    cfg,
    initial_pool_wrappers=initial_pool,
    benchmarks=benchmarks,
    initial_model=_init_for_train,
)

print(f'\nElapsed: {result["elapsed_sec"] / 60:.1f} min')
print(f'Final checkpoint: {result["checkpoint_path"]}')

In [ ]:
# Cell 8 — Optional Google Drive backup (Colab only).
#
# Mounts Drive once per session, then copies the finished SAC model +
# a tarball of the entire checkpoint directory to MyDrive. Silent no-op
# locally. Useful for overnight Colab runs — if the VM times out after
# Cell 7 saves its final checkpoint, this cell ensures a copy survives
# in Drive even if the Colab disk vanishes.
#
# Safe to skip. The committed checkpoint in `checkpoints/sac_run/` is
# already the ground truth for the repo.

if IS_COLAB:
    import shutil
    from datetime import datetime
    from google.colab import drive

    if not Path('/content/drive').is_mount():
        drive.mount('/content/drive')
    DRIVE_DIR = Path('/content/drive/MyDrive/connect4_sac_models')
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)

    stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    src   = Path(CHECKPOINT_DIR) / 'sac_model.keras'
    if src.exists():
        dst = DRIVE_DIR / f'sac_model_{stamp}.keras'
        shutil.copy(src, dst)
        print(f'✓ {dst}')

        tar_dst = DRIVE_DIR / f'sac_checkpoint_{stamp}.tar.gz'
        subprocess.run(
            ['tar', '-czf', str(tar_dst),
             '-C', str(Path(CHECKPOINT_DIR).parent),
             Path(CHECKPOINT_DIR).name],
            check=True,
        )
        print(f'✓ {tar_dst}')
    else:
        print(f'No {src} found — nothing to back up to Drive.')
else:
    print('Local run — skipping Drive backup (not relevant locally).')

In [ ]:
# Cell 9 — Plot training curves + eval history.
#
# Saves a 4-panel figure to `logs/sac_training_curves.png` that feeds
# directly into the Q7 report's SAC section.

import matplotlib.pyplot as plt
import numpy as np

log = result['log']
groups = np.array(log['group'])

fig, axes = plt.subplots(2, 2, figsize=(15, 9))

axes[0, 0].plot(groups, log['win_rate'], alpha=0.3, linewidth=0.8, label='per-group')
W = 20
if len(log['win_rate']) >= W:
    smooth = np.convolve(log['win_rate'], np.ones(W)/W, mode='valid')
    axes[0, 0].plot(groups[W-1:], smooth, linewidth=2, label=f'{W}-group avg')
axes[0, 0].axhline(0.5, color='grey', linestyle='--', linewidth=1)
axes[0, 0].set_title('Training-time win rate'); axes[0, 0].set_ylim(0, 1)
axes[0, 0].set_xlabel('Group'); axes[0, 0].legend(fontsize=8); axes[0, 0].grid(alpha=0.3)

axes[0, 1].plot(groups, log['q_loss'],      alpha=0.7, label='Q-loss (MSE)', linewidth=0.9)
axes[0, 1].plot(groups, log['policy_loss'], alpha=0.7, label='policy-loss',   linewidth=0.9)
axes[0, 1].set_title('SAC loss components'); axes[0, 1].set_xlabel('Group')
axes[0, 1].legend(fontsize=8); axes[0, 1].grid(alpha=0.3)

axes[1, 0].plot(groups, log['entropy'],     alpha=0.8, linewidth=0.9, label='policy entropy')
ax2 = axes[1, 0].twinx()
ax2.plot(groups, log['buffer_size'], color='orange', alpha=0.7, linewidth=0.9, label='buffer size')
axes[1, 0].set_title('Entropy + replay buffer fill'); axes[1, 0].set_xlabel('Group')
axes[1, 0].legend(loc='upper left', fontsize=8); axes[1, 0].grid(alpha=0.3)
ax2.legend(loc='upper right', fontsize=8)

if result['eval_history']:
    evgroups = [e['group'] for e in result['eval_history']]
    for name in benchmarks:
        rates = [e['results'][name]['win_rate'] for e in result['eval_history']]
        axes[1, 1].plot(evgroups, rates, marker='o', label=f'vs {name}')
    axes[1, 1].axhline(0.5, color='grey', linestyle='--', linewidth=1)
    axes[1, 1].set_ylim(0, 1); axes[1, 1].set_title('Periodic eval')
    axes[1, 1].set_xlabel('Group'); axes[1, 1].legend(fontsize=8); axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
fig_path = REPO_ROOT / 'logs' / 'sac_training_curves.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved figure to {fig_path}')

In [ ]:
# Cell 10 — Final head-to-head: SAC vs everything.
#
# 100 games against each opponent, alternating first player, with 4
# random warm-up moves so deterministic matchups don't collapse to
# 0/50/100% win rates.

from src.eval import play_match_parallel

sac_agent = SACAgent(result['model'], name='sac', greedy=True, use_tactics=True)

OPPONENTS = {
    'random':                  RandomAgent(),
    'minimax_d1':              MinimaxAgent(depth=1),
    'minimax_d3':              MinimaxAgent(depth=3),
    'minimax_d5':              MinimaxAgent(depth=5),
    'stiles_transformer_orig': ModelAgent(models['stiles_transformer_orig'], greedy=True, use_tactics=True),
    'stiles_cnn':              ModelAgent(models['stiles_cnn'],              greedy=True, use_tactics=True),
    'luke_cnn':                ModelAgent(models['luke_cnn'],                greedy=True, use_tactics=True),
    'luke_transformer':        ModelAgent(models['luke_transformer'],        greedy=True, use_tactics=True),
}
if 'zan_cnn' in models:
    OPPONENTS['zan_cnn'] = ModelAgent(models['zan_cnn'], greedy=True, use_tactics=True)
if 'zan_transformer' in models:
    OPPONENTS['zan_transformer'] = ModelAgent(models['zan_transformer'], greedy=True, use_tactics=True)

print('Final head-to-head (100 games each, random_init=4):\n')
rows = []
for name, opp in OPPONENTS.items():
    r = play_match_parallel(sac_agent, opp, n_games=100, random_init_moves=4, progress=False)
    rows.append((name, r.a_win_rate, r.draw_rate, r.b_win_rate, r.avg_length))
    print(f'  {name:30s}  W:{r.a_win_rate:6.1%}  D:{r.draw_rate:6.1%}  L:{r.b_win_rate:6.1%}  (avg {r.avg_length:.1f} moves)')

print(f'\nMean win rate: {sum(x[1] for x in rows) / len(rows):.1%}')

---

## ⚠️ Promote this run to the tournament model (manual, destructive)

The cell below copies the current checkpoint's model file into
`RL models/soft_actor_critic.keras`, **overwriting** whatever was there. This is
the canonical location the evaluation harness and tournament submission
pull from — make sure the model in this notebook is the one you want to
submit before uncommenting and running.

**Guardrails:**

- The cell is commented out so re-running all cells by accident cannot
  silently clobber the tournament model.
- It prints the old and new file sizes so you can sanity-check the copy.
- It does NOT delete the source — the checkpoint still lives at
  `checkpoints/sac_run/` and can be resumed from.

Uncomment the code in the next cell to run it.

In [ ]:
# # Cell 11 (commented) — promote this checkpoint to the tournament model.
# #
# # Copies checkpoints/sac_run/sac_model.keras  →  RL models/soft_actor_critic.keras
# # OVERWRITING the existing file at that destination.
#
# import shutil
#
# src = REPO_ROOT / 'checkpoints' / 'sac_run' / 'sac_model.keras'
# dst = REPO_ROOT / 'RL models' / 'soft_actor_critic.keras'
#
# assert src.exists(), f'Source not found: {src}'
# dst.parent.mkdir(parents=True, exist_ok=True)
#
# old_size = dst.stat().st_size if dst.exists() else 0
# shutil.copy(src, dst)
# new_size = dst.stat().st_size
#
# print(f'Replaced {dst.relative_to(REPO_ROOT)}')
# print(f'  old size: {old_size / 1024**2:6.1f} MB')
# print(f'  new size: {new_size / 1024**2:6.1f} MB')
# print(f'Commit and push to make this the tournament submission.')